# Nested CV BirdNET 训练

这个 notebook 是 `storm_petrel` vs `not_storm_petrel` 二分类 nested cross-validation 的训练入口。

默认 `RUN_FULL_NESTED_CV = False`，不会自动启动完整训练。确认检查结果后，再手动改成 `True`。

In [ ]:
from __future__ import annotations

import csv
import importlib.util
import os
import shutil
import subprocess
import sys
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

PROJECT_ROOT = Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if not SCRIPTS_DIR.exists():
    PROJECT_ROOT = Path.cwd().parents[1]
    SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from project_config import BINARY_LABELS_PATH as LABELS_PATH, DATASET_ROOT, FOLDS_PATH, MODEL_ROOT, NUMBA_CACHE_DIR, PROJECT_ROOT, TMP_DIR
SAMPLE_RATE = 48_000
SAMPLE_LENGTH = SAMPLE_RATE * 3

RUN_FULL_NESTED_CV = True
DEFAULT_EPOCHS = os.environ.get("BIRDNET_EPOCHS", "50")
DEFAULT_THREADS = os.environ.get("BIRDNET_THREADS", "8")

NUMBA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)
os.environ["NUMBA_CACHE_DIR"] = str(NUMBA_CACHE_DIR)
os.environ["TMP"] = str(TMP_DIR)
os.environ["TEMP"] = str(TMP_DIR)

pd.DataFrame([
    {"项目": "项目根目录", "路径": PROJECT_ROOT},
    {"项目": "fold 数据集", "路径": DATASET_ROOT},
    {"项目": "模型输出", "路径": MODEL_ROOT},
])

In [ ]:
modules = ["birdnet_analyzer", "tensorflow", "soundfile", "sklearn"]
env_rows = []
for name in modules:
    env_rows.append({"模块": name, "状态": "已安装" if importlib.util.find_spec(name) else "缺失"})
env_rows.append({"模块": "Python", "状态": sys.version.split()[0]})
pd.DataFrame(env_rows)

In [ ]:
def read_table(path: Path) -> pd.DataFrame:
    if path.is_file():
        return pd.read_csv(path)
    print(f"文件不存在，请先运行数据准备 pipeline: {path}")
    return pd.DataFrame()

labels = read_table(LABELS_PATH)
folds = read_table(FOLDS_PATH)

print(f"标签行数: {len(labels)}")
print(f"fold 行数: {len(folds)}")
if not labels.empty:
    display(labels.groupby(["target_label", "negative_type"], dropna=False).size().reset_index(name="count"))

In [ ]:
if folds.empty:
    print("没有 fold 表，先运行 scripts/data_processing/run_pipeline.py。")
else:
    fold_summary = (
        folds[folds["split"] == "outer_test"]
        .groupby(["outer_fold", "target_label"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    display(fold_summary)

    leakage_rows = []
    for outer_fold, part in folds.groupby("outer_fold"):
        train_files = set(part.loc[part["split"] == "outer_train", "source_file"])
        test_files = set(part.loc[part["split"] == "outer_test", "source_file"])
        leakage_rows.append({"outer_fold": outer_fold, "重叠 source_file 数": len(train_files & test_files)})
    display(pd.DataFrame(leakage_rows))

In [ ]:
@dataclass(frozen=True)
class HyperParameters:
    name: str
    learning_rate: str
    hidden_units: str
    dropout: str

HYPERPARAMETERS = [
    HyperParameters("baseline", "0.0001", "0", "0.0"),
    HyperParameters("hidden256_dropout02", "0.00005", "256", "0.2"),
    HyperParameters("hidden256_dropout03", "0.0001", "256", "0.3"),
    HyperParameters("hidden512_dropout03", "0.00005", "512", "0.3"),
]

pd.DataFrame([vars(item) for item in HYPERPARAMETERS])

In [ ]:
def run_birdnet_training(train_dir: Path, validation_dir: Path, output_prefix: Path, params: HyperParameters, cache_file: Path, cache_mode: str) -> bool:
    if not train_dir.is_dir() or not validation_dir.is_dir():
        print(f"跳过训练，目录不存在: train={train_dir}, validation={validation_dir}")
        return False

    command = [
        sys.executable, "-m", "birdnet_analyzer.train", str(train_dir),
        "-o", str(output_prefix),
        "--test_data", str(validation_dir),
        "--epochs", str(DEFAULT_EPOCHS),
        "--threads", str(DEFAULT_THREADS),
        "--learning_rate", params.learning_rate,
        "--hidden_units", params.hidden_units,
        "--dropout", params.dropout,
        "--model_save_mode", "replace",
        "--model_format", "tflite",
        "--cache_mode", cache_mode,
        "--cache_file", str(cache_file),
    ]
    output_prefix.parent.mkdir(parents=True, exist_ok=True)
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    print("运行训练命令:")
    print(" ".join(command))
    result = subprocess.run(command, cwd=PROJECT_ROOT)
    print(f"训练返回码: {result.returncode}")
    return result.returncode == 0

In [ ]:
def load_audio(path: Path):
    import soundfile as sf

    audio, sample_rate = sf.read(str(path), dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sample_rate != SAMPLE_RATE:
        print(f"跳过采样率不匹配文件: {path} ({sample_rate})")
        return None
    if len(audio) < SAMPLE_LENGTH:
        audio = np.pad(audio, (0, SAMPLE_LENGTH - len(audio)))
    return audio[:SAMPLE_LENGTH].astype("float32")


def predict_tflite(model_path: Path, data_dir: Path, positive_class: str = "storm_petrel") -> list[dict[str, str]]:
    if not model_path.is_file() or not data_dir.is_dir():
        print(f"没有可预测的模型或测试目录: model={model_path}, data={data_dir}")
        return []

    from tensorflow import lite as tflite

    interpreter = tflite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    rows: list[dict[str, str]] = []

    for folder in sorted(path for path in data_dir.iterdir() if path.is_dir()):
        true_label = 1 if folder.name == positive_class else 0
        for path in sorted(folder.glob("*.wav")):
            audio = load_audio(path)
            if audio is None:
                continue
            interpreter.resize_tensor_input(input_details["index"], [1, SAMPLE_LENGTH])
            interpreter.allocate_tensors()
            interpreter.set_tensor(input_details["index"], audio.reshape(1, SAMPLE_LENGTH))
            interpreter.invoke()
            score = float(interpreter.get_tensor(output_details["index"])[0][0])
            rows.append({"file": str(path), "class_folder": folder.name, "true_label": str(true_label), "score": f"{score:.8f}"})
    return rows

In [ ]:
def best_threshold(y_true: np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    from sklearn.metrics import f1_score

    best = (0.5, -1.0)
    for threshold in np.linspace(0.05, 0.95, 91):
        score = f1_score(y_true, scores >= threshold, zero_division=0)
        if score > best[1]:
            best = (float(threshold), float(score))
    return best


def write_evaluation(rows: list[dict[str, str]], output_prefix: Path, model_path: Path, data_dir: Path) -> dict[str, str]:
    if not rows:
        print("没有预测结果，跳过指标写出。")
        return {}

    from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

    output_prefix.parent.mkdir(parents=True, exist_ok=True)
    predictions_path = output_prefix.with_name(f"{output_prefix.name}_predictions.csv")
    metrics_path = output_prefix.with_name(f"{output_prefix.name}_metrics.csv")
    pd.DataFrame(rows).to_csv(predictions_path, index=False, encoding="utf-8")

    y_true = np.array([int(row["true_label"]) for row in rows])
    scores = np.array([float(row["score"]) for row in rows])
    threshold, _ = best_threshold(y_true, scores)
    y_pred_default = scores >= 0.5
    y_pred_opt = scores >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_opt, labels=[0, 1]).ravel()
    metrics = {
        "model": str(model_path), "data_dir": str(data_dir), "samples": len(rows),
        "precision_0_5": precision_score(y_true, y_pred_default, zero_division=0),
        "recall_0_5": recall_score(y_true, y_pred_default, zero_division=0),
        "f1_0_5": f1_score(y_true, y_pred_default, zero_division=0),
        "precision_opt": precision_score(y_true, y_pred_opt, zero_division=0),
        "recall_opt": recall_score(y_true, y_pred_opt, zero_division=0),
        "f1_opt": f1_score(y_true, y_pred_opt, zero_division=0),
        "auprc": average_precision_score(y_true, scores),
        "auroc": roc_auc_score(y_true, scores),
        "optimal_threshold": threshold,
        "true_positives": tp, "false_positives": fp, "true_negatives": tn, "false_negatives": fn,
    }
    pd.DataFrame([metrics]).to_csv(metrics_path, index=False, encoding="utf-8")
    return metrics

In [ ]:
def metric_from_evaluation(path: Path) -> float:
    if not path.is_file():
        print(f"找不到 BirdNET evaluation 文件，记为 0 分: {path}")
        return 0.0
    rows = pd.read_csv(path)
    if rows.empty:
        return 0.0
    selected = rows[rows.get("Class", "") == "storm_petrel"]
    row = selected.iloc[0] if not selected.empty else rows.iloc[0]
    for key in ["F1 Score (opt)", "F1 Score (0.5)", "f1_opt", "f1_0_5"]:
        if key in row and pd.notna(row[key]):
            return float(row[key])
    return 0.0


def run_inner_search(outer_fold: str) -> HyperParameters:
    outer_dataset = DATASET_ROOT / f"outer_{outer_fold}"
    outer_model_dir = MODEL_ROOT / f"outer_{outer_fold}"
    summary_rows = []
    inner_dirs = sorted(path for path in outer_dataset.glob("inner_*") if path.is_dir())
    if not inner_dirs:
        print(f"outer {outer_fold} 没有 inner fold，使用默认超参数。")
        return HYPERPARAMETERS[0]

    for params in HYPERPARAMETERS:
        scores = []
        for inner_dir in inner_dirs:
            output_prefix = outer_model_dir / "inner_search" / params.name / inner_dir.name / "model"
            cache_file = outer_model_dir / "cache" / f"{inner_dir.name}.npz"
            cache_mode = "save" if not cache_file.is_file() else "load"
            ok = run_birdnet_training(inner_dir / "train", inner_dir / "validation", output_prefix, params, cache_file, cache_mode)
            score = metric_from_evaluation(output_prefix.with_name(f"{output_prefix.name}_evaluation.csv")) if ok else 0.0
            scores.append(score)
            summary_rows.append({"outer_fold": outer_fold, "inner_fold": inner_dir.name.replace("inner_", ""), "params": params.name, "f1_opt": score})
        summary_rows.append({"outer_fold": outer_fold, "inner_fold": "mean", "params": params.name, "f1_opt": sum(scores) / len(scores) if scores else 0.0})

    summary = pd.DataFrame(summary_rows)
    summary_path = outer_model_dir / "inner_search_summary.csv"
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(summary_path, index=False, encoding="utf-8")
    best_name = summary[summary["inner_fold"] == "mean"].sort_values("f1_opt", ascending=False).iloc[0]["params"]
    return next(item for item in HYPERPARAMETERS if item.name == best_name)

In [ ]:
def run_outer_fold(outer_fold: str) -> None:
    outer_dataset = DATASET_ROOT / f"outer_{outer_fold}"
    outer_model_dir = MODEL_ROOT / f"outer_{outer_fold}"
    best_params = run_inner_search(outer_fold)

    selected = pd.DataFrame([{**vars(best_params), "outer_fold": outer_fold}])
    selected_path = outer_model_dir / "selected_hyperparameters.csv"
    selected_path.parent.mkdir(parents=True, exist_ok=True)
    selected.to_csv(selected_path, index=False, encoding="utf-8")

    final_prefix = outer_model_dir / "final_model" / "model"
    cache_file = outer_model_dir / "cache" / "final_train.npz"
    ok = run_birdnet_training(outer_dataset / "final_train", outer_dataset / "final_train", final_prefix, best_params, cache_file, "save" if not cache_file.is_file() else "load")
    if not ok:
        print(f"outer {outer_fold} 最终训练未成功，跳过测试评估。")
        return

    test_dir = outer_dataset / "test"
    model_path = final_prefix.with_suffix(".tflite")
    output_prefix = outer_model_dir / "outer_test"
    rows = predict_tflite(model_path, test_dir)
    metrics = write_evaluation(rows, output_prefix, model_path, test_dir)
    if metrics:
        shutil.copy2(output_prefix.with_name(f"{output_prefix.name}_predictions.csv"), outer_model_dir / "predictions.csv")
        shutil.copy2(output_prefix.with_name(f"{output_prefix.name}_metrics.csv"), outer_model_dir / "metrics.csv")

In [ ]:
outer_folds = sorted(path.name.replace("outer_", "") for path in DATASET_ROOT.glob("outer_*") if path.is_dir())
print("\u53ef\u8fd0\u884c\u7684\u5916\u5c42 folds:", outer_folds)

if RUN_FULL_NESTED_CV:
    for outer_fold in outer_folds:
        print()
        print(f"=== \u8fd0\u884c\u5916\u5c42 fold {outer_fold} ===")
        run_outer_fold(outer_fold)
    print()
    print(f"Nested CV \u8bad\u7ec3\u5b8c\u6210\uff0c\u7ed3\u679c\u4f4d\u4e8e {MODEL_ROOT}")
else:
    print("RUN_FULL_NESTED_CV=False\uff1a\u5f53\u524d\u53ea\u5b8c\u6210\u68c0\u67e5\uff0c\u4e0d\u542f\u52a8\u5b8c\u6574\u8bad\u7ec3\u3002")

In [ ]:
status_rows = []
for outer_fold in outer_folds:
    outer_dir = MODEL_ROOT / f"outer_{outer_fold}"
    status_rows.append({
        "outer_fold": outer_fold,
        "metrics": (outer_dir / "metrics.csv").is_file(),
        "predictions": (outer_dir / "predictions.csv").is_file(),
        "selected_hyperparameters": (outer_dir / "selected_hyperparameters.csv").is_file(),
    })
pd.DataFrame(status_rows)